# Week 4 — From predictions to “what should we do?”

Hi team — Week 3 gave metrics, plots, and a champion model. This notebook turns that into something you can **present**: who is high-risk, and a **simple** action table.

The scenario numbers are **illustrative** (not causal). Say that clearly in the deck.


### Imports

pandas + matplotlib. No sklearn.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
print("Libraries loaded")


### Load Week 3 outputs

Need `datasets/week3_predictions.csv` and `datasets/week3_model_metrics.csv`. Rerun Week 3 if either is missing.


In [ ]:
ROOT = Path(".")
PRED_IN = ROOT / "datasets" / "week3_predictions.csv"
METRIC_IN = ROOT / "datasets" / "week3_model_metrics.csv"
ACTION_OUT = ROOT / "datasets" / "week4_policy_actions.csv"

pred = pd.read_csv(PRED_IN, parse_dates=["date"])
metrics = pd.read_csv(METRIC_IN)

print("Pred shape:", pred.shape)
print("Metrics shape:", metrics.shape)
display(metrics)
display(pred.head())


### Pick champion for downstream use

Week 4 uses **lowest time-split RMSE** to choose `pred_ridge` vs `pred_rf` for the rest of this file. (Week 3 also prints a combined rule — if you want them identical, change this cell to match that rule.)

**Leave a note in the report** whichever rule you use.


In [ ]:
time_metrics = metrics[metrics["split"] == "time"].copy()
champion_row = time_metrics.sort_values("RMSE", ascending=True).iloc[0]
champion = "pred_rf" if champion_row["model"] == "RandomForest" else "pred_ridge"

print("Champion model:", champion_row["model"])
print("Prediction column:", champion)


### High-risk threshold

Flag months above the **75th percentile** of observed PM2.5 in this prediction slice. Easy to explain; swap for a regulatory limit later if units match.


In [ ]:
risk_threshold = pred["PM2_5"].quantile(0.75)
pred["high_risk_pred"] = pred[champion] >= risk_threshold
pred["high_risk_actual"] = pred["PM2_5"] >= risk_threshold

print("Risk threshold (75th percentile):", round(float(risk_threshold), 3))
print("Predicted high-risk rows:", int(pred["high_risk_pred"].sum()))
print("Actual high-risk rows:", int(pred["high_risk_actual"].sum()))


### High-risk precision / recall (rough)

Sanity check for slides: when we flag high risk, how often is reality high too? Not the main Week 3 metric, but helps the story.


In [ ]:
tp = int(((pred["high_risk_pred"] == 1) & (pred["high_risk_actual"] == 1)).sum())
fp = int(((pred["high_risk_pred"] == 1) & (pred["high_risk_actual"] == 0)).sum())
fn = int(((pred["high_risk_pred"] == 0) & (pred["high_risk_actual"] == 1)).sum())

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0

print("TP:", tp, "FP:", fp, "FN:", fn)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))


### Rank stations

Group by station: mean predicted PM2.5 and how often we flag high risk. Sort for “who needs attention first.”


In [ ]:
if "openaq_id" not in pred.columns:
    pred["openaq_id"] = "unknown"

station_risk = pred.groupby(["openaq_id"]).agg(mean_pred_pm25=(champion, "mean"), high_risk_months=("high_risk_pred", "sum"), total_months=("high_risk_pred", "count")).reset_index()
station_risk["high_risk_rate"] = station_risk["high_risk_months"] / station_risk["total_months"]
station_risk = station_risk.sort_values(["high_risk_rate", "mean_pred_pm25"], ascending=False)

display(station_risk.head(20))


### Simple scenario deltas

**Not causal.** We scale predictions by fixed factors (traffic / heating / combined) to show **example** reductions. Replace with real evidence when we have it.


In [ ]:
scenario = station_risk.copy()

# Simple planning assumptions (adjust later with better causal analysis)
scenario["pm25_if_traffic_control"] = scenario["mean_pred_pm25"] * 0.92
scenario["pm25_if_heating_control"] = scenario["mean_pred_pm25"] * 0.88
scenario["pm25_if_combined"] = scenario["mean_pred_pm25"] * 0.82

scenario["delta_traffic"] = scenario["mean_pred_pm25"] - scenario["pm25_if_traffic_control"]
scenario["delta_heating"] = scenario["mean_pred_pm25"] - scenario["pm25_if_heating_control"]
scenario["delta_combined"] = scenario["mean_pred_pm25"] - scenario["pm25_if_combined"]

display(scenario.head(20))


### Action labels

High / medium / monitor buckets from `high_risk_rate`. Tune 50% / 30% cutoffs if they feel off for Italy.


In [ ]:
def action_label(row):
    if row["high_risk_rate"] >= 0.50:
        return "High priority: combine traffic + heating actions"
    if row["high_risk_rate"] >= 0.30:
        return "Medium priority: seasonal traffic restrictions"
    return "Monitor: keep baseline controls"

scenario["recommended_action"] = scenario.apply(action_label, axis=1)

actions = scenario[["openaq_id", "mean_pred_pm25", "high_risk_rate", "delta_traffic", "delta_heating", "delta_combined", "recommended_action"]].copy()
actions = actions.sort_values(["high_risk_rate", "mean_pred_pm25"], ascending=False)
display(actions.head(30))


### Bar chart — top risky stations

Slide-friendly: which station ids show up most as high-risk in the test window.


In [ ]:
top_n = 15
plot_df = actions.head(top_n).copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["openaq_id"].astype(str), plot_df["high_risk_rate"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("High-risk rate")
plt.title("Top stations by predicted high-risk rate")
plt.tight_layout()
plt.show()


### Save `week4_policy_actions.csv`

Handoff for write-up or a simple dashboard.


In [ ]:
actions.to_csv(ACTION_OUT, index=False)
print("Saved:", ACTION_OUT)
print("Rows:", len(actions))


### Done

Reality-check with someone who knows winter heating + Po Valley weather before claiming policy impact.
